# Learning 4×4 sudoku with an Optyx `CMap`

This notebook turns the recurrent geometry proposed in [optyx#13](https://github.com/rel-int/optyx/issues/13) into a learning experiment similar to the [MapRNN sudoku demonstration](https://github.com/discopy/discopy/pull/416). Sixteen cell boxes exchange port-addressed messages with four row, four column and four square boxes. The Optyx `CMap` defines those routes; shared PyTorch modules learn the local cell and constraint updates.

Each cell has three incoming and three outgoing message ports plus two prediction ports. Each constraint has four incoming and four outgoing message ports. The two directions are separate, so the recurrent network has 96 edges and 192 memory wires; its 32 unpaired prediction wires form the boundary. The identity channels expose this backend-agnostic topology independently of the learning runtime.

In [1]:
import numpy as np
import torch
from torch.nn import functional as F

from optyx.channel import Diagram, qubit
from optyx.core.backends import DiscopyBackend
from optyx.interaction import Box, CMap
from optyx.qubits import Ket

torch.set_num_threads(1)
_ = torch.manual_seed(7)

## The recurrent map

The channel carried by a box can later be replaced without changing any edge. For this experiment the learnable interpreter below reads the same `CMap.partner` relation, making the categorical wiring the single source of truth.

In [2]:
size = 4
n_cells = size ** 2
n_constraints = 3 * size


def cell_box(index):
    ports = qubit ** 8
    return Box(
        f"cell_{index}", qubit ** 3, qubit ** 5, Diagram.id(ports))


def constraint_box(kind, index):
    ports = qubit ** 8
    return Box(
        f"{kind}_{index}", qubit ** 4, qubit ** 4, Diagram.id(ports))


cells = [cell_box(index) for index in range(n_cells)]
constraints = [
    constraint_box(kind, index)
    for kind in ("row", "column", "square")
    for index in range(size)
]
boxes = cells + constraints

In [3]:
def memberships(row, column):
    square = 2 * (row // 2) + column // 2
    square_position = 2 * (row % 2) + column % 2
    return (
        (n_cells + row, column),
        (n_cells + size + column, row),
        (n_cells + 2 * size + square, square_position),
    )


edges = []
for cell in range(n_cells):
    row, column = divmod(cell, size)
    for slot, (constraint, position) in enumerate(
            memberships(row, column)):
        edges.append(((cell, slot), (constraint, size + position)))
        edges.append(((cell, 3 + slot), (constraint, position)))

sudoku = CMap(boxes, edges)
summary = {
    "boxes": len(sudoku.boxes),
    "edges": len(sudoku.edges),
    "prediction_wires": len(sudoku.boundary),
    "memory_wires": len(sudoku.memory),
}
assert summary == {
    "boxes": 28, "edges": 96,
    "prediction_wires": 32, "memory_wires": 192}
summary

{'boxes': 28, 'edges': 96, 'prediction_wires': 32, 'memory_wires': 192}

## A deterministic sudoku dataset

There are only 288 completed 4×4 sudoku grids, so we enumerate the complete corpus. We split by completed grid before hiding clues, preventing the same solution from appearing in both sets. Every puzzle keeps eight clues and is accepted only when those clues identify a unique member of the corpus.

In [4]:
def groups():
    rows = [tuple(row * size + column for column in range(size))
            for row in range(size)]
    columns = [tuple(row * size + column for row in range(size))
               for column in range(size)]
    squares = [
        tuple((2 * block_row + row) * size
              + 2 * block_column + column
              for row in range(2) for column in range(2))
        for block_row in range(2) for block_column in range(2)
    ]
    return rows + columns + squares


sudoku_groups = groups()
peers = [set() for _ in range(n_cells)]
for group in sudoku_groups:
    for cell in group:
        peers[cell].update(set(group) - {cell})


def sudoku_solutions():
    result, grid = [], [0] * n_cells

    def visit(cell):
        if cell == n_cells:
            result.append(tuple(grid))
            return
        for value in range(1, size + 1):
            if all(grid[peer] != value for peer in peers[cell]):
                grid[cell] = value
                visit(cell + 1)
        grid[cell] = 0

    visit(0)
    return np.asarray(result)


solutions = sudoku_solutions()
assert solutions.shape == (288, n_cells)

In [5]:
def unique_puzzle(solution, random):
    while True:
        clue_cells = random.choice(n_cells, n_cells // 2, replace=False)
        matches = np.all(
            solutions[:, clue_cells] == solution[clue_cells], axis=1)
        if matches.sum() == 1:
            puzzle = np.zeros(n_cells, dtype=int)
            puzzle[clue_cells] = solution[clue_cells]
            return puzzle


def make_dataset(source, repeats, random):
    clues, targets = [], []
    for solution in source:
        for _ in range(repeats):
            clues.append(unique_puzzle(solution, random))
            targets.append(solution - 1)
    return (
        torch.tensor(np.stack(clues), dtype=torch.long),
        torch.tensor(np.stack(targets), dtype=torch.long),
    )


random = np.random.default_rng(7)
order = random.permutation(len(solutions))
train_solutions = solutions[order[:224]]
test_solutions = solutions[order[224:]]
train_clues, train_targets = make_dataset(train_solutions, 4, random)
test_clues, test_targets = make_dataset(test_solutions, 2, random)
dataset_summary = {
    "solutions": len(solutions),
    "training_puzzles": len(train_clues),
    "held_out_puzzles": len(test_clues),
    "clues_per_puzzle": int((train_clues > 0).sum(1).unique().item()),
}
dataset_summary

{'solutions': 288,
 'training_puzzles': 896,
 'held_out_puzzles': 128,
 'clues_per_puzzle': 8}

## A MapRNN interpreter for the `CMap`

One cell module is shared by all sixteen cells and one constraint module by all twelve constraints. At every round, a cell combines its clue embedding, hidden state and three incoming messages. Its three outgoing messages are scattered into the constraint ports specified by `sudoku.partner`; constraint replies are gathered along the same routes. No neighbourhood pooling or hard-coded row, column or square computation appears in the model.

In [6]:
def mlp(domain, codomain, width):
    return torch.nn.Sequential(
        torch.nn.Linear(domain, width),
        torch.nn.Tanh(),
        torch.nn.Linear(width, codomain),
    )


class SudokuMapRNN(torch.nn.Module):
    """Shared local updates routed by an Optyx combinatorial map."""

    def __init__(self, cmap, dimension=16, n_rounds=8):
        super().__init__()
        self.dimension = dimension
        self.n_rounds = n_rounds
        route = []
        for cell in range(n_cells):
            cell_route = []
            for slot in range(3):
                constraint, position = cmap.partner[(cell, 3 + slot)]
                cell_route.append((constraint - n_cells, position))
            route.append(cell_route)
        route = torch.tensor(route, dtype=torch.long)
        expected = [(constraint, position)
                    for constraint in range(n_constraints)
                    for position in range(size)]
        assert sorted(map(tuple, route.reshape(-1, 2).tolist())) == expected
        self.register_buffer("constraint_index", route[..., 0])
        self.register_buffer("position_index", route[..., 1])
        self.embedding = torch.nn.Embedding(size + 1, dimension)
        self.cell = mlp(5 * dimension, 4 * dimension, 8 * dimension)
        self.constraint = mlp(
            4 * dimension, 4 * dimension, 8 * dimension)
        self.readout = torch.nn.Linear(dimension, size)

    def forward(self, clues):
        batch = clues.shape[0]
        incoming = torch.zeros(
            batch, n_cells, 3, self.dimension, device=clues.device)
        hidden = torch.zeros(
            batch, n_cells, self.dimension, device=clues.device)
        embedded = self.embedding(clues)
        for _ in range(self.n_rounds):
            cell_input = torch.cat(
                (embedded, hidden, incoming.flatten(2)), dim=-1)
            cell_output = self.cell(cell_input).reshape(
                batch, n_cells, 4, self.dimension)
            messages, hidden = cell_output[:, :, :3], cell_output[:, :, 3]
            constraint_input = torch.zeros(
                batch, n_constraints, size, self.dimension,
                device=clues.device)
            constraint_input[
                :, self.constraint_index, self.position_index] = messages
            constraint_output = self.constraint(
                constraint_input.flatten(2)).reshape(
                    batch, n_constraints, size, self.dimension)
            incoming = constraint_output[
                :, self.constraint_index, self.position_index]
        return self.readout(hidden)

## Training and held-out evaluation

Cross entropy is evaluated only on hidden cells; copying a visible clue cannot reduce the loss. Decoding restores the clues before measuring hidden-cell accuracy, exact-grid accuracy and the fraction of predictions satisfying all twelve sudoku constraints.

In [7]:
def predictions_for(model, clues):
    predictions = model(clues).argmax(-1)
    return torch.where(clues > 0, clues - 1, predictions)


def is_valid(grid):
    digits = set(range(size))
    return all(set(grid[list(group)].tolist()) == digits
               for group in sudoku_groups)


def evaluate(model, clues, targets):
    with torch.no_grad():
        logits = model(clues)
        hidden = clues == 0
        predictions = torch.where(
            clues > 0, clues - 1, logits.argmax(-1))
        return {
            "loss": F.cross_entropy(
                logits[hidden], targets[hidden]).item(),
            "hidden_cell_accuracy": (
                predictions[hidden] == targets[hidden]
            ).float().mean().item(),
            "exact_grid_accuracy": (
                predictions == targets).all(1).float().mean().item(),
            "valid_grid_accuracy": np.mean([
                is_valid(grid) for grid in predictions]),
        }


model = SudokuMapRNN(sudoku)
initial_metrics = evaluate(model, test_clues, test_targets)
initial_metrics

{'loss': 1.3878284692764282,
 'hidden_cell_accuracy': 0.2431640625,
 'exact_grid_accuracy': 0.0,
 'valid_grid_accuracy': 0.0}

In [8]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
training_steps = 300
batch_size = 64
checkpoints = {1, 50, 100, 200, training_steps}
history = []
for step in range(1, training_steps + 1):
    batch = torch.randint(0, len(train_clues), (batch_size,))
    logits = model(train_clues[batch])
    hidden = train_clues[batch] == 0
    loss = F.cross_entropy(logits[hidden], train_targets[batch][hidden])
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step in checkpoints:
        metrics = evaluate(model, test_clues, test_targets)
        history.append((step, loss.item(), metrics["loss"]))

final_metrics = evaluate(model, test_clues, test_targets)
assert final_metrics["loss"] < initial_metrics["loss"]
assert final_metrics["hidden_cell_accuracy"] > 0.75
history, final_metrics

([(1, 1.3879660367965698, 1.389206886291504),
  (50, 0.18860521912574768, 0.20152875781059265),
  (100, 0.01153749693185091, 0.09547462314367294),
  (200, 0.0003919259470421821, 0.0055108265951275826),
  (300, 0.00556592782959342, 0.03801686316728592)],
 {'loss': 0.03801686316728592,
  'hidden_cell_accuracy': 0.9921875,
  'exact_grid_accuracy': 0.9765625,
  'valid_grid_accuracy': 0.9765625})

In [9]:
example = 0
example_prediction = predictions_for(
    model, test_clues[example:example + 1])[0] + 1
{
    "clues": test_clues[example].reshape(size, size).tolist(),
    "prediction": example_prediction.reshape(size, size).tolist(),
    "target": (test_targets[example] + 1).reshape(size, size).tolist(),
}

{'clues': [[3, 0, 0, 0], [4, 2, 3, 1], [1, 0, 4, 0], [0, 4, 0, 0]],
 'prediction': [[3, 1, 2, 4], [4, 2, 3, 1], [1, 3, 4, 2], [2, 4, 1, 3]],
 'target': [[3, 1, 2, 4], [4, 2, 3, 1], [1, 3, 4, 2], [2, 4, 1, 3]]}

## Stream and approximate fixpoint semantics

The learned interpreter uses a fixed number of synchronous rounds, matching `sudoku.unroll(n_steps)`: the stream semantics copies local updates through time and routes every output to its partner's next input. Independently, `CMap.fix` exposes the approximate stationary boundary of the same recurrent shape. The complete sudoku map has 192 memory qubits, so the small executable probe below checks the API without attempting its exponentially large dense fixed point.

In [10]:
wire = Box("wire", qubit, qubit ** 2, Diagram.id(qubit ** 3))
probe = CMap([wire], [((0, 1), (0, 2))])
fixed = probe.fix(
    Ket(1), Ket(0) @ Ket(0), n_steps=2,
    backend=DiscopyBackend())
assert np.allclose(fixed.density_matrix, [[0, 0], [0, 1]])

This run demonstrates learning over the exact Optyx interaction topology, but it deliberately does not claim differentiability through `Diagram.to_tensor`. Parametrised channel boxes and backend-neutral gradients through Quimb, Cotengra, JAX or PyTorch contractions remain a separate tensor-backend task; when available, they can replace this interpreter without changing the dataset or `CMap` wiring.